# 🌍 ClimateSense AI — ML Training Notebook
**Live Urban Heat Island Monitoring, Climate Intelligence and ML-Based Temperature Prediction**
Third-year ML project · *Global Warming and Climate Change* · topic: **Urban Heat-Island Reduction**

### What this notebook does (and does not do)
| ✅ It does | ❌ It does not |
|---|---|
| Downloads **real** archived weather-model data (Open-Meteo Historical Forecast API) for 6 Indian cities | invent, simulate or randomly generate any weather value |
| Trains **Linear Regression, SVR (RBF), Random Forest** to predict *tomorrow's maximum temperature* | claim that this daily model *proves* greenhouse-gas causation |
| Evaluates with **MAE, MSE, R²** on a chronological validation and an untouched test set | hard-code any score — every number below is computed when you run the cells |
| Builds 1991–2020 **ERA5-Land climate normals** to compute temperature anomalies | |
| Explains the model (feature importance, permutation importance, SHAP) | |
| Exports everything the FastAPI backend needs | |

> **Weather forecasting ≠ climate-change analysis.** Predicting tomorrow's temperature is a *weather* task (days).
> Climate change is the long-term (decades) shift in the statistics of weather. The website shows both, but keeps them separate:
> the ML model forecasts weather; the NOAA CO₂/CH₄/N₂O and NASA GISTEMP pages show long-term climate evidence.

### How to run
1. Open this file in Google Colab → **Runtime → Run all** (takes roughly 10–25 minutes, mostly API downloads and model tuning).
2. The last cell downloads `climatesense_artifacts.zip`. Unzip it into the project root so that `backend/models/` and `data/` are filled.
3. The notebook needs internet access (Colab has it) — it calls the public Open-Meteo API, which needs no key.

## Section 1 — Environment setup
We import the libraries and fix random seeds so that results are **reproducible**: running the notebook twice on the same data gives the same models.

In [ ]:
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    """Import a package; pip-install it first if it is missing."""
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name or import_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name], check=False)
        return importlib.import_module(import_name)

for imp, pip in [("pandas", None), ("numpy", None), ("matplotlib", None), ("seaborn", None),
                 ("sklearn", "scikit-learn"), ("requests", None), ("joblib", None)]:
    ensure(imp, pip)
try:
    shap = ensure("shap")
    SHAP_OK = True
except Exception as exc:          # SHAP is optional: the notebook falls back gracefully
    SHAP_OK = False
    print("SHAP could not be imported -> SHAP cells will use the built-in explainer:", exc)

import os, json, time, random, hashlib, platform, warnings, zipfile
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import requests, joblib, sklearn
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42                                   # one seed used everywhere
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path("climatesense_output")          # everything we create lives here
RAW_DIR, PROC_DIR = ROOT / "data" / "raw", ROOT / "data" / "processed"
MODEL_DIR = ROOT / "backend" / "models"     # <- the folder the FastAPI backend reads
FIG_DIR = MODEL_DIR / "figures"
for d in (RAW_DIR, PROC_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| SHAP", shap.__version__ if SHAP_OK else "n/a")

### 1.1 Shared feature-engineering code
**Why this matters:** a very common bug in ML projects is *train/serve skew* — the features computed for training differ slightly from those computed in the live app, so the model silently gets wrong inputs.
To make that impossible, the notebook writes out the **exact same Python files** that the backend imports (`feature_engineering.py`, `explain_core.py`). Training and live inference therefore run identical code.

In [ ]:
FE_SOURCE = r'''
"""ClimateSense AI - shared feature engineering (SINGLE SOURCE OF TRUTH).

This exact file is used in two places:
  1. the Google Colab training notebook (it is written to disk by the notebook), and
  2. the FastAPI backend (live inference).

Keeping one file guarantees that training features and live-inference features are
computed by identical code (no train/serve skew).

Prediction task
---------------
Cutoff day D  ->  predict the MAXIMUM 2 m air temperature of day D+1.
Only information from days <= D is used. The only forward-looking column is
`target_tmax`, which is NEVER part of FEATURES.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

FEATURE_MODULE_VERSION = "1.0.0"
TIMEZONE = "Asia/Kolkata"

# City name -> coordinates (decimal degrees). Used for training AND serving.
CITIES = {
    "Chennai": {"lat": 13.0827, "lon": 80.2707},
    "Delhi": {"lat": 28.6139, "lon": 77.2090},
    "Mumbai": {"lat": 19.0760, "lon": 72.8777},
    "Bengaluru": {"lat": 12.9716, "lon": 77.5946},
    "Hyderabad": {"lat": 17.3850, "lon": 78.4867},
    "Kolkata": {"lat": 22.5726, "lon": 88.3639},
}

# Hourly variables that the model needs (names follow the Open-Meteo API).
MODEL_HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "surface_pressure",
    "cloud_cover",
    "shortwave_radiation",
]

# Expected units from the Open-Meteo API (validated in training and live).
EXPECTED_UNITS = {
    "temperature_2m": "°C",
    "relative_humidity_2m": "%",
    "precipitation": "mm",
    "wind_speed_10m": "km/h",
    "surface_pressure": "hPa",
    "cloud_cover": "%",
    "shortwave_radiation": "W/m²",
}

# Model input columns, in this exact order.
FEATURES = [
    "tmax_d0",
    "tmin_d0",
    "tmax_d1",
    "tmax_d2",
    "tmin_d1",
    "rh_mean_d0",
    "precip_d0",
    "precip_sum3",
    "wind_mean_d0",
    "solar_mj_d0",
    "pressure_mean_d0",
    "cloud_mean_d0",
    "tmax_roll3",
    "tmax_roll7",
    "doy_sin",
    "doy_cos",
    "latitude",
    "longitude",
]
TARGET = "target_tmax"

FEATURE_INFO = {
    "tmax_d0": ("Max temperature, cutoff day", "°C"),
    "tmin_d0": ("Min temperature, cutoff day", "°C"),
    "tmax_d1": ("Max temperature, 1 day before cutoff", "°C"),
    "tmax_d2": ("Max temperature, 2 days before cutoff", "°C"),
    "tmin_d1": ("Min temperature, 1 day before cutoff", "°C"),
    "rh_mean_d0": ("Mean relative humidity, cutoff day", "%"),
    "precip_d0": ("Precipitation total, cutoff day", "mm"),
    "precip_sum3": ("Precipitation total, last 3 days", "mm"),
    "wind_mean_d0": ("Mean wind speed (10 m), cutoff day", "km/h"),
    "solar_mj_d0": ("Shortwave solar radiation total, cutoff day", "MJ/m²"),
    "pressure_mean_d0": ("Mean surface pressure, cutoff day", "hPa"),
    "cloud_mean_d0": ("Mean cloud cover, cutoff day", "%"),
    "tmax_roll3": ("3-day rolling mean of max temperature", "°C"),
    "tmax_roll7": ("7-day rolling mean of max temperature", "°C"),
    "doy_sin": ("Seasonality: sin(2π·day-of-year/365) of target day", "-"),
    "doy_cos": ("Seasonality: cos(2π·day-of-year/365) of target day", "-"),
    "latitude": ("Latitude of the city", "°"),
    "longitude": ("Longitude of the city", "°"),
}

# Project-defined heat categories (NOT official IMD warnings).
ANOMALY_ELEVATED_C = 1.5
HEAT_LEVELS = [
    {"level": 0, "label": "Normal", "color": "#2dd4bf",
     "rule": "Predicted max < normal + 1.5 °C and below the 90th percentile"},
    {"level": 1, "label": "Elevated", "color": "#facc15",
     "rule": "Anomaly >= +1.5 °C but below the 90th percentile of the city's seasonal window"},
    {"level": 2, "label": "High", "color": "#fb923c",
     "rule": "Predicted max >= 90th percentile of the city's seasonal window"},
    {"level": 3, "label": "Severe", "color": "#ef4444",
     "rule": "Predicted max >= 95th percentile of the city's seasonal window"},
]
HEAT_DISCLAIMER = (
    "Project-defined heat categories derived from a 1991-2020 ERA5-Land climatology. "
    "They are NOT official India Meteorological Department (IMD) warnings."
)


class FeatureError(ValueError):
    """Raised when features cannot be built honestly (missing / incomplete data)."""


def day_of_year_365(dates) -> np.ndarray:
    """Day of year on a fixed 365-day calendar (29 Feb is mapped onto 28 Feb)."""
    idx = pd.DatetimeIndex(dates)
    doy = np.asarray(idx.dayofyear, dtype=int).copy()
    leap_after_feb = np.asarray(idx.is_leap_year) & (doy >= 60)
    return np.where(leap_after_feb, doy - 1, doy)


def validate_units(units: dict | None) -> list[str]:
    """Return a list of unit problems (empty list = all units as expected)."""
    problems = []
    if not units:
        return ["no units returned by API"]
    for var, expected in EXPECTED_UNITS.items():
        got = units.get(var)
        if got is None:
            problems.append(f"{var}: unit missing")
        elif got.replace(" ", "") != expected.replace(" ", ""):
            problems.append(f"{var}: expected {expected}, got {got}")
    return problems


def hourly_to_daily(hourly: pd.DataFrame, min_hours: int = 22) -> pd.DataFrame:
    """Aggregate hourly API data into local-calendar-day records.

    `hourly` needs a `time` column (local time, as returned with timezone=Asia/Kolkata)
    plus the MODEL_HOURLY_VARIABLES. Days with fewer than `min_hours` valid temperature
    values are dropped (never filled / invented).
    """
    missing = [c for c in ["time", *MODEL_HOURLY_VARIABLES] if c not in hourly.columns]
    if missing:
        raise FeatureError(f"hourly data lacks columns: {missing}")
    df = hourly[["time", *MODEL_HOURLY_VARIABLES]].copy()
    df["time"] = pd.to_datetime(df["time"])
    df["date"] = df["time"].dt.normalize()
    g = df.groupby("date")
    daily = pd.DataFrame(
        {
            "tmax": g["temperature_2m"].max(),
            "tmin": g["temperature_2m"].min(),
            "rh_mean": g["relative_humidity_2m"].mean(),
            "precip_sum": g["precipitation"].sum(min_count=1),
            "wind_mean": g["wind_speed_10m"].mean(),
            "pressure_mean": g["surface_pressure"].mean(),
            "cloud_mean": g["cloud_cover"].mean(),
            # hourly mean W/m2 x 3600 s = J/m2 per hour; sum -> MJ/m2
            "solar_mj": g["shortwave_radiation"].sum(min_count=1) * 3600.0 / 1e6,
            "n_hours": g["temperature_2m"].count(),
        }
    )
    daily = daily[daily["n_hours"] >= min_hours]
    return daily.reset_index()


def make_feature_frame(daily: pd.DataFrame, latitude: float, longitude: float) -> pd.DataFrame:
    """Build model features (and the training target) from one city's daily table.

    Row for date D contains ONLY information from days <= D. `target_tmax` is the
    max temperature of D+1 (NaN for the most recent day) and is never a feature.
    """
    d = daily.sort_values("date").drop_duplicates("date").set_index("date")
    if d.empty:
        raise FeatureError("daily table is empty")
    # calendar-continuous index so lags/rolling windows are true calendar days
    d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq="D"))

    f = pd.DataFrame(index=d.index)
    f["tmax_d0"] = d["tmax"]
    f["tmin_d0"] = d["tmin"]
    f["tmax_d1"] = d["tmax"].shift(1)
    f["tmax_d2"] = d["tmax"].shift(2)
    f["tmin_d1"] = d["tmin"].shift(1)
    f["rh_mean_d0"] = d["rh_mean"]
    f["precip_d0"] = d["precip_sum"]
    f["precip_sum3"] = d["precip_sum"].rolling(3, min_periods=3).sum()
    f["wind_mean_d0"] = d["wind_mean"]
    f["solar_mj_d0"] = d["solar_mj"]
    f["pressure_mean_d0"] = d["pressure_mean"]
    f["cloud_mean_d0"] = d["cloud_mean"]
    f["tmax_roll3"] = d["tmax"].rolling(3, min_periods=3).mean()
    f["tmax_roll7"] = d["tmax"].rolling(7, min_periods=7).mean()

    target_dates = f.index + pd.Timedelta(days=1)
    doy = day_of_year_365(target_dates)
    f["doy_sin"] = np.sin(2 * np.pi * doy / 365.0)
    f["doy_cos"] = np.cos(2 * np.pi * doy / 365.0)
    f["latitude"] = float(latitude)
    f["longitude"] = float(longitude)

    f[TARGET] = d["tmax"].shift(-1)  # <- the only forward-looking column
    f.index.name = "date"
    f = f.reset_index()
    f["target_date"] = f["date"] + pd.Timedelta(days=1)
    return f[["date", "target_date", *FEATURES, TARGET]]


def feature_row_for_date(daily: pd.DataFrame, latitude: float, longitude: float, cutoff_date) -> pd.DataFrame:
    """One-row DataFrame (columns == FEATURES) for live inference at `cutoff_date`."""
    frame = make_feature_frame(daily, latitude, longitude)
    cutoff = pd.Timestamp(cutoff_date).normalize()
    row = frame[frame["date"] == cutoff]
    if row.empty:
        raise FeatureError(f"no daily record available for cutoff date {cutoff.date()}")
    x = row[FEATURES].astype(float)
    bad = [c for c in FEATURES if not np.isfinite(x.iloc[0][c])]
    if bad:
        raise FeatureError(f"missing / non-finite features for {cutoff.date()}: {bad}")
    return x.reset_index(drop=True)


def classify_heat(predicted: float, normal: float, p90: float, p95: float) -> dict:
    """Project-defined heat category (see HEAT_LEVELS)."""
    anomaly = float(predicted) - float(normal)
    if predicted >= p95:
        level = 3
    elif predicted >= p90:
        level = 2
    elif anomaly >= ANOMALY_ELEVATED_C:
        level = 1
    else:
        level = 0
    info = dict(HEAT_LEVELS[level])
    info["anomaly_c"] = round(anomaly, 2)
    info["disclaimer"] = HEAT_DISCLAIMER
    return info


def feature_schema_entries() -> list[dict]:
    return [
        {"name": n, "label": FEATURE_INFO[n][0], "unit": FEATURE_INFO[n][1]} for n in FEATURES
    ]
'''
EXPLAIN_SOURCE = r'''
"""ClimateSense AI - lightweight local explanation (shared by notebook and API).

`permutation_shapley` estimates Shapley-style attributions for ONE prediction using
only `model.predict` (no extra dependencies):

  * baseline  = training-set mean of every feature (stored in feature_schema.json)
  * sampling  = random feature orderings; features are switched from baseline to the
                real value one by one and the change in the model output is credited
                to the feature that was switched.
  * property  = contributions always sum EXACTLY to  f(x) - f(baseline).

It is model-agnostic and works for Linear Regression, SVR and Random Forest.
The notebook compares it against the `shap` library as a sanity check.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def permutation_shapley(model, x_row: pd.Series, baseline: pd.Series, features: list[str],
                        n_permutations: int = 120, seed: int = 42) -> dict:
    rng = np.random.default_rng(seed)
    n = len(features)
    x = x_row[features].to_numpy(dtype=float)
    b = baseline[features].to_numpy(dtype=float)

    rows = []
    orders = []
    for _ in range(n_permutations):
        order = rng.permutation(n)
        orders.append(order)
        cur = b.copy()
        rows.append(cur.copy())  # all-baseline row
        for j in order:
            cur[j] = x[j]
            rows.append(cur.copy())
    batch = pd.DataFrame(np.vstack(rows), columns=features)
    preds = np.asarray(model.predict(batch), dtype=float).reshape(n_permutations, n + 1)

    contrib = np.zeros(n)
    for p, order in enumerate(orders):
        deltas = np.diff(preds[p])
        contrib[order] += deltas
    contrib /= n_permutations

    base_value = float(preds[:, 0].mean())
    prediction = float(preds[:, -1].mean())
    return {
        "base_value": base_value,
        "prediction": prediction,
        "contributions": {f: float(c) for f, c in zip(features, contrib)},
        "method": f"permutation-sampling Shapley ({n_permutations} orderings, training-mean baseline)",
    }
'''

Path('feature_engineering.py').write_text(FE_SOURCE.lstrip('\n'), encoding='utf-8')
Path('explain_core.py').write_text(EXPLAIN_SOURCE.lstrip('\n'), encoding='utf-8')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import feature_engineering as fe
import explain_core
importlib.reload(fe)

FE_SHA256 = hashlib.sha256(Path('feature_engineering.py').read_text(encoding='utf-8').replace('\r\n', '\n').encode()).hexdigest()
print('feature module version:', fe.FEATURE_MODULE_VERSION, '| sha256:', FE_SHA256[:16], '...')
print('Cities       :', list(fe.CITIES))
print('Features (%d):' % len(fe.FEATURES), fe.FEATURES)
print('Target       :', fe.TARGET, '(maximum temperature of the NEXT day)')

## Section 2 — Data collection (real data only)

### Data product used for training
**Open-Meteo Historical Forecast API** (`historical-forecast-api.open-meteo.com`). It stores *archived weather-model forecasts* (roughly from 2022 onward), stitched together from the first hours of successive model runs.
This is deliberate: the **live website uses the Open-Meteo Forecast API**, i.e. the same family of weather-model products, so the training inputs resemble the live inputs as closely as possible.

> ⚠️ **Honest limitation:** these are *model-derived* values, not readings from a station thermometer. The "target" we predict is therefore the model-estimated maximum temperature, not an official IMD station observation.

For every city we download **hourly** values of: `temperature_2m, relative_humidity_2m, precipitation, wind_speed_10m, surface_pressure, cloud_cover, shortwave_radiation` and aggregate them into daily records with the shared function `fe.hourly_to_daily` (so training and live aggregation are identical).

In [ ]:
HF_URL   = "https://historical-forecast-api.open-meteo.com/v1/forecast"   # archived NWP forecasts
ERA5_URL = "https://archive-api.open-meteo.com/v1/archive"                # ERA5 / ERA5-Land reanalysis

TRAIN_START = date(2022, 1, 1)                    # archive begins around 2022
TRAIN_END   = date.today() - timedelta(days=2)    # latest days may still be incomplete, so stop 2 days back
RETRIEVED_AT = datetime.now(timezone.utc)

class ApiError(RuntimeError):
    pass

def get_json(url, params, retries=6):
    """GET with retries/back-off. Raises ApiError - we never continue with partial or made-up data."""
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 200:
                js = r.json()
                if isinstance(js, dict) and js.get("error"):
                    raise ApiError(js.get("reason", "API reported an error"))
                return js
            last = f"HTTP {r.status_code}: {r.text[:200]}"
            if r.status_code == 429:
                time.sleep(min(90, 15 * (attempt + 1)))       # rate limited -> wait
            elif r.status_code >= 500:
                time.sleep(4 * (attempt + 1))
            else:
                break                                          # 4xx = our request is wrong; retrying will not help
        except (requests.RequestException, ValueError) as exc:
            last = str(exc)
            time.sleep(4 * (attempt + 1))
    raise ApiError(f"request failed: {last}")

def year_chunks(start, end):
    cur = start
    while cur <= end:
        stop = min(date(cur.year, 12, 31), end)
        yield cur, stop
        cur = stop + timedelta(days=1)

def fetch_city_hourly(city):
    c = fe.CITIES[city]
    parts, units, grid = [], None, None
    for a, b in year_chunks(TRAIN_START, TRAIN_END):
        js = get_json(HF_URL, dict(
            latitude=c["lat"], longitude=c["lon"], start_date=a.isoformat(), end_date=b.isoformat(),
            hourly=",".join(fe.MODEL_HOURLY_VARIABLES), timezone=fe.TIMEZONE,
            temperature_unit="celsius", wind_speed_unit="kmh", precipitation_unit="mm"))
        df = pd.DataFrame(js["hourly"])
        units = units or js.get("hourly_units")
        grid = grid or {"grid_latitude": js.get("latitude"), "grid_longitude": js.get("longitude"),
                        "elevation_m": js.get("elevation")}
        parts.append(df)
        time.sleep(1.0)                                        # be polite to the free API
    out = pd.concat(parts, ignore_index=True)
    out.insert(0, "city", city)
    return out, units, grid

hourly_parts, units_by_city, grid_by_city = [], {}, {}
for city in fe.CITIES:
    t0 = time.time()
    df, units, grid = fetch_city_hourly(city)
    problems = fe.validate_units(units)
    if problems:
        raise ApiError(f"{city}: unexpected units -> {problems}")        # stop instead of mixing units
    hourly_parts.append(df); units_by_city[city] = units; grid_by_city[city] = grid
    print(f"{city:<10} {len(df):>7,} hourly rows | grid cell {grid['grid_latitude']:.3f},{grid['grid_longitude']:.3f} "
          f"| elevation {grid['elevation_m']} m | {time.time() - t0:.0f}s")

hourly_raw = pd.concat(hourly_parts, ignore_index=True)
hourly_raw.to_csv(RAW_DIR / "hourly_raw.csv.gz", index=False)           # raw dataset saved as CSV (gzip)
print("\nTotal hourly rows:", f"{len(hourly_raw):,}", "| saved ->", RAW_DIR / "hourly_raw.csv.gz")

In [ ]:
# Coverage report: what period does each city REALLY cover (null rows are not counted as data)?
hourly_raw["time"] = pd.to_datetime(hourly_raw["time"])
valid = hourly_raw.dropna(subset=["temperature_2m"])
coverage = valid.groupby("city").agg(first_valid=("time", "min"), last_valid=("time", "max"), valid_hours=("time", "count"))
coverage["requested_hours"] = (pd.Timestamp(TRAIN_END) - pd.Timestamp(TRAIN_START)).days * 24 + 24
coverage["coverage_pct"] = (100 * coverage["valid_hours"] / coverage["requested_hours"]).round(1)
display(coverage)

OBS_START, OBS_END = valid["time"].min().date(), valid["time"].max().date()
print(f"Observation period actually available: {OBS_START} -> {OBS_END}")

### 2.1 Save the dataset metadata
A dataset without documentation is hard to trust. We store *where* the data came from, *when* it was downloaded, *what period* it covers, the *units*, the *record count* and the *cities*.

In [ ]:
dataset_meta = {
    "name": "ClimateSense AI training dataset (hourly, raw)",
    "source": "Open-Meteo Historical Forecast API (archived weather-model forecasts)",
    "source_url": HF_URL,
    "documentation": "https://open-meteo.com/en/docs/historical-forecast-api",
    "license": "CC BY 4.0 - Open-Meteo.com (weather data); free tier for non-commercial use",
    "retrieval_date_utc": RETRIEVED_AT.isoformat(timespec="seconds"),
    "requested_period": f"{TRAIN_START} to {TRAIN_END}",
    "observation_period": f"{OBS_START} to {OBS_END}",
    "timezone": fe.TIMEZONE,
    "cities": {c: {**fe.CITIES[c], **grid_by_city[c]} for c in fe.CITIES},
    "variables": fe.MODEL_HOURLY_VARIABLES,
    "units": units_by_city[next(iter(fe.CITIES))],
    "hourly_records": int(len(hourly_raw)),
    "records_per_city": hourly_raw.groupby("city").size().to_dict(),
    "nature_of_values": "model-derived (archived NWP forecasts), not station observations",
}
(RAW_DIR / "dataset_metadata.json").write_text(json.dumps(dataset_meta, indent=2, ensure_ascii=False))
print(json.dumps({k: dataset_meta[k] for k in ["source", "retrieval_date_utc", "observation_period", "hourly_records", "units"]},
                 indent=2, ensure_ascii=False))

## Section 3 — Data preprocessing
Steps (all are *inspected and reported*, nothing is silently changed):
1. **Duplicate removal** — the same `(city, time)` must appear once.
2. **Missing-value analysis** — we look at nulls before deciding anything; days with too few hours are *dropped*, never filled with invented numbers.
3. **Date parsing** and **unit validation**.
4. **Outlier inspection** — we *look* for strange values but **do not remove genuine extreme-heat events**: heatwaves are exactly what this project is about.
5. **Feature engineering** with the shared module — using only information available at the prediction cutoff.

In [ ]:
# 3.1 Duplicates + date parsing
n0 = len(hourly_raw)
hourly = hourly_raw.drop_duplicates(subset=["city", "time"]).copy()
hourly["time"] = pd.to_datetime(hourly["time"], errors="raise")
print(f"Duplicate hourly rows removed: {n0 - len(hourly)}")

# 3.2 Missing-value analysis (percent of null hourly values)
miss = hourly.groupby("city")[fe.MODEL_HOURLY_VARIABLES].apply(lambda g: g.isna().mean() * 100).round(3)
print("\nMissing hourly values (%):")
display(miss)

# 3.3 Unit / plausibility validation (physical ranges for India; counts of values outside)
plausible = {"temperature_2m": (-5, 52), "relative_humidity_2m": (0, 100), "precipitation": (0, 200),
             "wind_speed_10m": (0, 250), "surface_pressure": (800, 1100), "cloud_cover": (0, 100),
             "shortwave_radiation": (0, 1500)}
rows = []
for var, (lo, hi) in plausible.items():
    s = hourly[var].dropna()
    rows.append({"variable": var, "unit": fe.EXPECTED_UNITS[var], "min": s.min(), "max": s.max(),
                 "plausible_range": f"{lo}..{hi}", "outside_range": int(((s < lo) | (s > hi)).sum())})
display(pd.DataFrame(rows))

In [ ]:
# 3.4 Aggregate to daily records with the SHARED function (days with < 22 valid hours are dropped, never filled)
daily_parts = []
for city, g in hourly.groupby("city"):
    d = fe.hourly_to_daily(g.drop(columns="city"))
    d.insert(0, "city", city)
    daily_parts.append(d)
daily = pd.concat(daily_parts, ignore_index=True)

expected_days = (pd.Timestamp(OBS_END) - pd.Timestamp(OBS_START)).days + 1
summary = daily.groupby("city").agg(days=("date", "count"), first=("date", "min"), last=("date", "max"))
summary["days_missing_or_incomplete"] = expected_days - summary["days"]
display(summary)
daily.to_csv(PROC_DIR / "daily_aggregated.csv", index=False)

# 3.5 Outlier inspection: robust z-score of daily max temperature vs the city's own month median.
daily["month"] = daily["date"].dt.month
med = daily.groupby(["city", "month"])["tmax"].transform("median")
mad = (daily["tmax"] - med).abs().groupby([daily["city"], daily["month"]]).transform("median").clip(lower=0.3)
daily["robust_z"] = 0.6745 * (daily["tmax"] - med) / mad
flag = daily[daily["robust_z"].abs() > 3.5]
print(f"\nDays flagged by the robust z-score (|z|>3.5): {len(flag)} of {len(daily)}  -> INSPECTED, NOT REMOVED")
display(flag.sort_values("tmax", ascending=False).head(10)[["city", "date", "tmax", "tmin", "robust_z"]])
print("Hottest days in the dataset (genuine extremes are kept):")
display(daily.sort_values("tmax", ascending=False).head(5)[["city", "date", "tmax"]])
daily = daily.drop(columns=["month", "robust_z"])

In [ ]:
# 3.6 Feature engineering (target = NEXT day's maximum temperature; features use days <= cutoff only)
frames = []
for city, g in daily.groupby("city"):
    c = fe.CITIES[city]
    ff = fe.make_feature_frame(g.drop(columns="city"), c["lat"], c["lon"])
    ff.insert(0, "city", city)
    frames.append(ff)
features_all = pd.concat(frames, ignore_index=True)

model_df = features_all.dropna(subset=fe.FEATURES + [fe.TARGET]).reset_index(drop=True)
print(f"Rows before dropping incomplete windows: {len(features_all):,}")
print(f"Rows usable for modelling              : {len(model_df):,}  "
      f"(dropped {len(features_all) - len(model_df)}: first 6 days per city lack a 7-day window, last day lacks a target)")

# Hard safety checks: the target must never be among the features, and it must be tomorrow's tmax.
assert fe.TARGET not in fe.FEATURES
check = model_df.merge(daily[["city", "date", "tmax"]].rename(columns={"date": "target_date", "tmax": "tmax_check"}),
                       on=["city", "target_date"])
assert np.allclose(check[fe.TARGET], check["tmax_check"]), "target is not the next-day maximum!"
print("✔ Leakage check passed: target == tmax of the following day; no forward-looking feature.")

model_df.to_csv(PROC_DIR / "model_dataset.csv", index=False)
print("\nDescriptive statistics:")
display(model_df[fe.FEATURES + [fe.TARGET]].describe().T.round(3))

In [ ]:
# 3.7 Update the dataset metadata with the daily/model row counts
dataset_meta.update({
    "daily_records": int(len(daily)), "model_rows": int(len(model_df)),
    "model_rows_per_city": model_df.groupby("city").size().to_dict(),
    "first_model_date": str(model_df["date"].min().date()), "last_model_date": str(model_df["date"].max().date()),
    "daily_aggregation": {"day": "local calendar day (Asia/Kolkata)", "min_valid_hours": 22,
                          "tmax/tmin": "max/min of hourly temperature_2m", "precipitation": "sum of hourly values",
                          "humidity/wind/pressure/cloud": "mean of hourly values",
                          "solar": "sum of hourly shortwave radiation converted to MJ/m2"},
    "omitted_features": ["Dew point and apparent temperature were not used: not needed for the minimum feature set."],
})
(RAW_DIR / "dataset_metadata.json").write_text(json.dumps(dataset_meta, indent=2, ensure_ascii=False))
print("metadata updated:", RAW_DIR / "dataset_metadata.json")

## Section 4 — Exploratory Data Analysis (EDA)
EDA means *looking at the data before modelling*. Every figure below is followed by what to look for, and an automatic text summary computed from the real data (so the statements always match what you see).

In [ ]:
def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

palette = dict(zip(fe.CITIES, sns.color_palette("husl", len(fe.CITIES))))

# 4.1 Temperature trend plots (7-day rolling mean of daily max temperature)
fig, ax = plt.subplots(figsize=(13, 5))
for city, g in daily.groupby("city"):
    g = g.sort_values("date")
    ax.plot(g["date"], g["tmax"].rolling(7, min_periods=4).mean(), label=city, color=palette[city], lw=1.3)
ax.set(title="Daily maximum temperature (7-day rolling mean)", xlabel="Date", ylabel="Tmax (°C)")
ax.legend(ncol=3)
save_fig("eda_01_temperature_trends")

**What it reveals:** the *seasonal cycle* (pre-monsoon peak, monsoon dip, winter low) and differences between cities. Delhi swings the most (continental climate); coastal cities (Chennai, Mumbai) vary less. Year-to-year differences hint at how unusual a given season was — but 4 years are far too short to say anything about *climate change*.

In [ ]:
# 4.2 City-wise comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = daily.groupby("city")["tmax"].median().sort_values().index
sns.boxplot(data=daily, x="city", y="tmax", order=order, palette=palette, ax=axes[0])
axes[0].set(title="City-wise distribution of daily Tmax", xlabel="", ylabel="Tmax (°C)")
sns.boxplot(data=daily.assign(dtr=daily["tmax"] - daily["tmin"]), x="city", y="dtr", order=order, palette=palette, ax=axes[1])
axes[1].set(title="Diurnal temperature range (Tmax − Tmin)", xlabel="", ylabel="°C")
save_fig("eda_02_city_comparison")

# 4.3 Historical distributions
g = sns.displot(data=daily, x="tmax", col="city", col_wrap=3, kde=True, height=3, color="#0d9488", bins=30)
g.set_axis_labels("Tmax (°C)", "days")
g.fig.suptitle("Distribution of daily maximum temperature by city", y=1.03, fontweight="bold")
save_fig("eda_03_distributions")

**What it reveals:** the *median* and *spread* of each city's Tmax. Wide boxes = large seasonal range. Points above the whiskers are the hottest days (kept in the data). A large diurnal range (Delhi, Hyderabad) indicates dry air; a small one (Mumbai, Kolkata) indicates humid air. Distributions that are two-humped (bimodal) reflect hot-dry and cooler-wet seasons.

In [ ]:
# 4.4 Correlation heatmap
corr_cols = [c for c in fe.FEATURES if c not in ("latitude", "longitude")] + [fe.TARGET]
corr = model_df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation between features and next-day Tmax")
save_fig("eda_04_correlation_heatmap")
top = corr[fe.TARGET].drop(fe.TARGET).abs().sort_values(ascending=False).head(5)
print("Features most correlated (|r|) with next-day Tmax:\n", top.round(3).to_string())

**What it reveals:** today's temperature measures (`tmax_d0`, `tmax_roll3`, `tmax_roll7`) correlate strongly with tomorrow's — *persistence* is a strong baseline in weather. Humidity and cloud cover correlate **negatively** (humid/cloudy air limits daytime heating), solar radiation positively. Strongly inter-correlated features (e.g. the three temperature averages) are *redundant*; that matters for interpreting linear coefficients but not for tree models.

In [ ]:
# 4.5 Seasonal trend analysis
monthly = daily.assign(month=daily["date"].dt.month).groupby(["city", "month"])["tmax"].mean().reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
for city, g in monthly.groupby("city"):
    ax.plot(g["month"], g["tmax"], marker="o", label=city, color=palette[city])
ax.set(title="Mean daily Tmax by calendar month (dataset period)", xlabel="Month", ylabel="°C", xticks=range(1, 13))
ax.legend(ncol=3)
save_fig("eda_05_seasonal")

# 4.6 Missing-value visualisation
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(miss, annot=True, fmt=".2f", cmap="YlOrRd", cbar_kws={"label": "% missing"}, ax=ax)
ax.set_title("Missing hourly values (%) by city and variable")
save_fig("eda_06_missing_values")
print("Total missing hourly values:", int(hourly[fe.MODEL_HOURLY_VARIABLES].isna().sum().sum()))

**What they reveal:** the seasonal plot shows *when* each city peaks (e.g. April–May pre-monsoon in most cities). The missing-value map confirms data quality: a (near-)white heat-map means (almost) nothing was missing; any coloured cell tells us which city/variable to be careful with.

In [ ]:
# 4.7 Scatter plots of important weather variables vs next-day Tmax
sample = model_df.sample(min(4000, len(model_df)), random_state=SEED)
pairs = [("tmax_d0", "Today's Tmax (°C)"), ("rh_mean_d0", "Mean humidity today (%)"),
         ("solar_mj_d0", "Solar radiation today (MJ/m²)"), ("cloud_mean_d0", "Mean cloud cover today (%)")]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (col, lab) in zip(axes.ravel(), pairs):
    sns.scatterplot(data=sample, x=col, y=fe.TARGET, hue="city", palette=palette, s=10, alpha=0.5, ax=ax, legend=(col == "tmax_d0"))
    ax.set(xlabel=lab, ylabel="Next-day Tmax (°C)")
fig.suptitle("Key weather variables vs next-day maximum temperature", fontweight="bold")
save_fig("eda_07_scatter")

hot = daily.loc[daily["tmax"].idxmax()]
warm_month = monthly.loc[monthly["tmax"].idxmax()]
print("AUTOMATIC OBSERVATIONS (computed from the data)")
print(f" • Highest daily Tmax in the dataset: {hot['tmax']:.1f} °C in {hot['city']} on {hot['date'].date()}")
print(f" • City with the highest median Tmax : {daily.groupby('city')['tmax'].median().idxmax()}")
print(f" • Warmest city-month on average     : {warm_month['city']}, month {int(warm_month['month'])} ({warm_month['tmax']:.1f} °C)")
print(f" • Strongest single predictor (|r|)  : {top.index[0]} (r = {corr.loc[top.index[0], fe.TARGET]:.2f})")

**What it reveals:** a tight diagonal in the first panel = temperature persistence. A downward cloud for humidity/cloud cover and upward cloud for solar radiation show *physical* relationships the model can use. Overlapping coloured clouds from different cities show why latitude/longitude are included.

## Section 5 — Train / validation / test split (chronological, no leakage)

* **Time never flows backwards.** We sort all unique *dates* and cut them into **70 % train → 15 % validation → 15 % test**.
* Splitting by **unique dates** (not rows) means every city's record for a given date goes into the *same* partition.
* A **1-day gap (embargo)** between partitions makes sure the target of the last training row is not an input of the first validation row.
* **Time-aware cross-validation** inside the training set: each fold trains on the past and validates on the *next* block of dates (expanding window). We never shuffle.
* **Imputers and scalers are fitted inside the scikit-learn Pipeline**, so inside every CV fold they learn only from that fold's training part, and finally only from the training set.
* The **test set is not touched** until the model has been chosen.

In [ ]:
dates = np.sort(model_df["date"].unique())
n = len(dates); i1, i2 = int(n * 0.70), int(n * 0.85)
train_dates, val_dates, test_dates = dates[:i1], dates[i1 + 1:i2], dates[i2 + 1:]     # 1-day embargo at each boundary

train_df = model_df[model_df["date"].isin(train_dates)].reset_index(drop=True)
val_df   = model_df[model_df["date"].isin(val_dates)].reset_index(drop=True)
test_df  = model_df[model_df["date"].isin(test_dates)].reset_index(drop=True)

X_train, y_train = train_df[fe.FEATURES], train_df[fe.TARGET]
X_val,   y_val   = val_df[fe.FEATURES],   val_df[fe.TARGET]
X_test,  y_test  = test_df[fe.FEATURES],  test_df[fe.TARGET]

split_info = {}
for name, part, dts in [("train", train_df, train_dates), ("validation", val_df, val_dates), ("test", test_df, test_dates)]:
    split_info[name] = {"start": str(pd.Timestamp(dts[0]).date()), "end": str(pd.Timestamp(dts[-1]).date()),
                        "n_dates": int(len(dts)), "n_rows": int(len(part)), "share_pct": round(100 * len(part) / len(model_df), 1)}
display(pd.DataFrame(split_info).T)

# Safety checks
assert train_df["date"].max() < val_df["date"].min() < val_df["date"].max() < test_df["date"].min()
assert set(train_dates).isdisjoint(val_dates) and set(val_dates).isdisjoint(test_dates)
for dts in (train_dates, val_dates, test_dates):                 # every city must be in the same date partition
    assert model_df[model_df["date"].isin(dts)].groupby("date")["city"].nunique().max() <= len(fe.CITIES)
print("✔ Chronological order, disjoint partitions and same-date-same-partition verified.")

def date_blocked_time_splits(dates_col, n_splits=5, embargo=1):
    """Expanding-window CV over UNIQUE DATES: train on the past, validate on the next block."""
    u = np.sort(dates_col.unique()); k = len(u) // (n_splits + 1); arr = dates_col.to_numpy()
    for s in range(1, n_splits + 1):
        tr, va = u[: s * k], u[s * k + embargo: (s + 1) * k]
        yield np.where(np.isin(arr, tr))[0], np.where(np.isin(arr, va))[0]

cv_splits = list(date_blocked_time_splits(train_df["date"], n_splits=5))
for i, (tr, va) in enumerate(cv_splits, 1):
    print(f"CV fold {i}: train {train_df['date'].iloc[tr].min().date()} → {train_df['date'].iloc[tr].max().date()}  |  "
          f"validate {train_df['date'].iloc[va].min().date()} → {train_df['date'].iloc[va].max().date()}")

## Section 6 — Model implementation

### The algorithms in simple language
**Persistence baseline.** "Tomorrow's maximum = today's maximum." Weather has memory, so this simple rule is surprisingly hard to beat. Any real model must do better, otherwise it adds no value.

**1. Linear Regression.** Fits one straight-line formula:  `Tmax_tomorrow = b0 + b1·x1 + b2·x2 + …`. It learns the weights `b` that minimise the squared error. *Strength:* fast, easy to explain. *Weakness:* can only model straight-line relationships. Needs **scaling** so that features with different units (hPa vs °C) are comparable.

**2. Support Vector Regression (SVR, RBF kernel).** Fits a smooth curve that stays within a tolerance tube of width **ε** around the data; points outside the tube are penalised with strength **C**. The **RBF kernel** lets it model curved relationships by measuring similarity between days; **γ** controls how local that similarity is. *Strength:* captures non-linear patterns. *Weakness:* sensitive to scaling and hyper-parameters; slower on huge data. We scale inputs *and* the target.

**3. Random Forest Regressor.** Builds many decision trees on random subsets of rows and features and **averages** them. Each tree asks questions like "was today above 36 °C?". *Strength:* handles non-linear interactions, robust, gives feature importance, needs no scaling. *Weakness:* cannot extrapolate beyond the temperatures seen in training; larger model file.

### Why Pipelines?
A `Pipeline` bundles *imputer → scaler → model* into one object. It guarantees the same preprocessing at training time and live prediction time, and prevents leakage because each step is fitted only on training data.

### Hyper-parameter tuning
`GridSearchCV` tries many settings, scores each with the **time-aware CV on the training data only**, and keeps the best (lowest MAE). Validation and test sets are not used for tuning.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def make_linear():
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("lr", LinearRegression())])

def make_svr():
    inner = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                      ("svr", SVR(kernel="rbf", cache_size=500))])
    return TransformedTargetRegressor(regressor=inner, transformer=StandardScaler())     # target scaled too

def make_rf():
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("rf", RandomForestRegressor(random_state=SEED, n_jobs=-1))])

CANDIDATES = {
    "Linear Regression": (make_linear(), {}),
    "SVR (RBF)": (make_svr(), {"regressor__svr__C": [1, 10, 30],
                               "regressor__svr__epsilon": [0.05, 0.1],
                               "regressor__svr__gamma": ["scale", 0.01]}),
    "Random Forest": (make_rf(), {"rf__n_estimators": [300], "rf__max_depth": [8, 14, None],
                                  "rf__min_samples_leaf": [2, 5], "rf__max_features": [0.5, 1.0]}),
}

def regression_metrics(y_true, y_pred):
    return {"mae": float(mean_absolute_error(y_true, y_pred)),
            "mse": float(mean_squared_error(y_true, y_pred)),
            "r2": float(r2_score(y_true, y_pred))}

fitted, search_info = {}, {}
for name, (est, grid) in CANDIDATES.items():
    t0 = time.time()
    gs = GridSearchCV(est, grid, scoring="neg_mean_absolute_error", cv=cv_splits, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)                                    # tuning sees ONLY training data
    fitted[name] = gs.best_estimator_
    i = gs.best_index_
    search_info[name] = {"best_params": {k: (None if v is None else v) for k, v in gs.best_params_.items()},
                         "cv_mae_mean": float(-gs.cv_results_["mean_test_score"][i]),
                         "cv_mae_std": float(gs.cv_results_["std_test_score"][i]),
                         "n_candidates": int(len(gs.cv_results_["params"])), "n_cv_folds": len(cv_splits),
                         "fit_seconds": round(time.time() - t0, 1)}
    print(f"{name:<18} CV-MAE {search_info[name]['cv_mae_mean']:.3f} ± {search_info[name]['cv_mae_std']:.3f} °C | "
          f"best params: {gs.best_params_} | {search_info[name]['fit_seconds']}s")

## Section 7 — Evaluation metrics
For every model we compute three standard regression metrics (all in the units of the target):

| Metric | Meaning | Units | Better is |
|---|---|---|---|
| **MAE** — Mean Absolute Error | average size of the error, `mean(|actual − predicted|)` | °C | lower |
| **MSE** — Mean Squared Error | average of squared errors; punishes large misses (heat-wave misses!) | °C² | lower |
| **R²** — Coefficient of determination | share of the variation in Tmax the model explains (1 = perfect, 0 = no better than predicting the mean) | – | higher |

**Selection rule:** the best model is the ML algorithm with the **lowest validation MAE**. Only after that decision do we compute test-set scores; the test set never influences the choice. The *persistence baseline* ("tomorrow = today") is reported next to the models, so we can see whether machine learning really adds value.

In [ ]:
ML_NAMES = list(CANDIDATES)
persistence = lambda df: df["tmax_d0"].to_numpy()

results = {}
for name, model in fitted.items():
    results[name] = {"train": regression_metrics(y_train, model.predict(X_train)),
                     "validation": regression_metrics(y_val, model.predict(X_val))}
baseline = {"validation": regression_metrics(y_val, persistence(val_df))}

# ---- MODEL SELECTION: validation MAE only -------------------------------------------------
best_name = min(ML_NAMES, key=lambda n: results[n]["validation"]["mae"])
best_model = fitted[best_name]
print(f"Selected model (lowest validation MAE): {best_name}  ({results[best_name]['validation']['mae']:.3f} °C)")

# ---- Only now evaluate on the untouched test set --------------------------------------------
for name, model in fitted.items():
    results[name]["test"] = regression_metrics(y_test, model.predict(X_test))
baseline["test"] = regression_metrics(y_test, persistence(test_df))

rows = []
for name in ML_NAMES + ["Persistence baseline"]:
    r = baseline if name.startswith("Persistence") else results[name]
    row = {"Model": name}
    if name in results: row["Train MAE"] = results[name]["train"]["mae"]
    for part in ("validation", "test"):
        for m in ("mae", "mse", "r2"):
            row[f"{part[:3].title()} {m.upper()}"] = r[part][m]
    rows.append(row)
comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.round(3).style.highlight_min(subset=["Val MAE", "Val MSE", "Tes MAE", "Tes MSE"], color="#bbf7d0")
        .highlight_max(subset=["Val R2", "Tes R2"], color="#bbf7d0"))

imp_val  = 100 * (1 - results[best_name]["validation"]["mae"] / baseline["validation"]["mae"])
imp_test = 100 * (1 - results[best_name]["test"]["mae"] / baseline["test"]["mae"])
print(f"\nBest model vs persistence baseline: validation MAE {imp_val:+.1f}% , test MAE {imp_test:+.1f}% "
      f"({'better' if imp_test > 0 else 'WORSE'} than 'tomorrow = today').")
if imp_test <= 0:
    print("⚠ The ML model does not beat persistence on the test set - report this honestly.")

In [ ]:
# Model-wise evaluation chart
names = ML_NAMES + ["Persistence"]
def pick(part, m): return [results[n][part][m] for n in ML_NAMES] + [baseline[part][m]]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.arange(len(names)); w = 0.38
for ax, (m, title) in zip(axes, [("mae", "MAE (°C) ↓"), ("mse", "MSE (°C²) ↓"), ("r2", "R² ↑")]):
    ax.bar(x - w / 2, pick("validation", m), w, label="Validation", color="#14b8a6")
    ax.bar(x + w / 2, pick("test", m), w, label="Test", color="#0f766e")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15); ax.set_title(title); ax.legend()
fig.suptitle(f"Model-wise evaluation — best by validation MAE: {best_name}", fontweight="bold")
save_fig("eval_01_model_comparison")

# Actual vs predicted (test set)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True)
lims = [min(y_test.min(), 15), y_test.max() + 1]
for ax, name in zip(axes, ML_NAMES):
    p = fitted[name].predict(X_test)
    ax.scatter(y_test, p, s=6, alpha=0.4, color="#0d9488"); ax.plot(lims, lims, "k--", lw=1)
    ax.set(title=f"{name}  (test MAE {results[name]['test']['mae']:.2f} °C)", xlabel="Actual next-day Tmax (°C)", ylabel="Predicted (°C)")
save_fig("eval_02_actual_vs_predicted")

In [ ]:
# Residual plots for the selected model (test set). residual = actual - predicted
pred_test = best_model.predict(X_test); resid_test = y_test.to_numpy() - pred_test
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
sns.histplot(resid_test, bins=40, kde=True, color="#0d9488", ax=axes[0]); axes[0].axvline(0, color="k", ls="--")
axes[0].set(title="Residual distribution", xlabel="actual − predicted (°C)")
axes[1].scatter(pred_test, resid_test, s=6, alpha=0.4, color="#0d9488"); axes[1].axhline(0, color="k", ls="--")
axes[1].set(title="Residuals vs predicted", xlabel="predicted (°C)", ylabel="residual (°C)")
sns.boxplot(data=pd.DataFrame({"city": test_df["city"], "residual": resid_test}), x="city", y="residual", palette=palette, ax=axes[2])
axes[2].axhline(0, color="k", ls="--"); axes[2].set(title="Residuals by city", xlabel="")
fig.suptitle(f"Residual analysis — {best_name}", fontweight="bold")
save_fig("eval_03_residuals")

per_city_mae = {c: float(mean_absolute_error(g[fe.TARGET], best_model.predict(g[fe.FEATURES]))) for c, g in test_df.groupby("city")}
print("Test MAE per city (°C):", {k: round(v, 2) for k, v in per_city_mae.items()})
print("Mean residual (bias): %.3f °C | residual std: %.3f °C" % (resid_test.mean(), resid_test.std()))

# Empirical prediction interval: 5th/95th percentile of VALIDATION residuals (calibration), coverage checked on TEST
resid_val = y_val.to_numpy() - best_model.predict(X_val)
lo_off, hi_off = float(np.quantile(resid_val, 0.05)), float(np.quantile(resid_val, 0.95))
coverage = float(np.mean((y_test.to_numpy() >= pred_test + lo_off) & (y_test.to_numpy() <= pred_test + hi_off)))
prediction_interval = {"level": 0.90, "lower_offset": lo_off, "upper_offset": hi_off, "calibrated_on": "validation residuals (5th-95th percentile)",
                       "test_coverage": coverage, "method": "empirical residual quantiles; interval = prediction + offset"}
print(f"90% empirical interval: [{lo_off:+.2f}, {hi_off:+.2f}] °C around the prediction | observed coverage on test: {coverage:.1%}")

In [ ]:
MODEL_DESCRIPTIONS = {
    "Linear Regression": "Straight-line model on scaled features (imputer → StandardScaler → LinearRegression).",
    "SVR (RBF)": "Support Vector Regression with RBF kernel; inputs and target standardised; C, epsilon, gamma tuned with time-aware CV.",
    "Random Forest": "Ensemble of decision trees averaged; depth, leaf size, trees and max_features tuned with time-aware CV.",
}
FIGURES = []
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, (np.ndarray,)): return o.tolist()
    if isinstance(o, (pd.Timestamp, date)): return str(o)[:10]
    return o

hist_counts, hist_edges = np.histogram(resid_test, bins=30)
rng = np.random.default_rng(SEED)
sel = rng.choice(len(test_df), size=min(400, len(test_df)), replace=False)
metrics_doc = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "units": {"mae": "°C", "mse": "°C²", "r2": "dimensionless"},
    "selection_rule": "lowest validation MAE among the three ML algorithms; test set untouched until selection",
    "best_model": {"name": best_name, "validation": results[best_name]["validation"], "test": results[best_name]["test"]},
    "models": {n: {**results[n], **search_info[n], "description": MODEL_DESCRIPTIONS[n]} for n in ML_NAMES},
    "baseline_persistence": {**baseline, "description": "Predict tomorrow's Tmax = today's Tmax"},
    "improvement_over_persistence_pct": {"validation_mae": imp_val, "test_mae": imp_test},
    "split": split_info,
    "per_city_test_mae": per_city_mae,
    "prediction_interval": prediction_interval,
    "test_scatter": [{"actual": round(float(y_test.iloc[i]), 2), "predicted": round(float(pred_test[i]), 2), "city": test_df["city"].iloc[i]} for i in sel],
    "residual_histogram": {"bin_edges": hist_edges.round(3).tolist(), "counts": hist_counts.tolist(), "mean": float(resid_test.mean()), "std": float(resid_test.std())},
}
(MODEL_DIR / "model_metrics.json").write_text(json.dumps(jsonable(metrics_doc), indent=2, ensure_ascii=False))
print("saved:", MODEL_DIR / "model_metrics.json")

## Section 8 — Explainable AI (XAI)
A forecast is only useful if people can ask **"why?"**. We use four complementary views:

1. **Random Forest impurity importance** — how much each feature reduced error inside the forest's trees (fast, but biased towards features with many distinct values).
2. **Permutation importance (on the test set)** — shuffle one feature at a time and measure how much the MAE gets worse. Model-agnostic and measured on unseen data.
3. **SHAP values** — game-theory-based contribution of each feature to each individual prediction, in °C.
4. **Local explanation** for single predictions — which features pushed *this* forecast up or down.

All values below are computed from the actual fitted models.

In [ ]:
from sklearn.inspection import permutation_importance

# 8.1 Random Forest impurity importance (the RF candidate is always fitted, even if it is not the best model)
rf_model = fitted["Random Forest"].named_steps["rf"]
rf_imp = pd.Series(rf_model.feature_importances_, index=fe.FEATURES).sort_values(ascending=False)

# 8.2 Permutation importance of the SELECTED model on the test set
pi = permutation_importance(best_model, X_test, y_test, scoring="neg_mean_absolute_error",
                            n_repeats=10, random_state=SEED, n_jobs=-1)
perm = pd.DataFrame({"mean": pi.importances_mean, "std": pi.importances_std}, index=fe.FEATURES).sort_values("mean", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
rf_imp[::-1].plot.barh(ax=axes[0], color="#14b8a6"); axes[0].set_title("Random Forest impurity importance"); axes[0].set_xlabel("importance")
axes[1].barh(perm.index[::-1], perm["mean"][::-1], xerr=perm["std"][::-1], color="#0f766e")
axes[1].set_title(f"Permutation importance — {best_name} (test set)"); axes[1].set_xlabel("increase in MAE when shuffled (°C)")
save_fig("xai_01_feature_importance")
print("Top-5 by permutation importance:\n", perm.head(5).round(4).to_string())

In [ ]:
# 8.3 SHAP (global + local). Tree models use the exact TreeExplainer; others use KernelExplainer.
X_expl = X_test.sample(min(150, len(X_test)), random_state=SEED).reset_index(drop=True)
shap_values, shap_base, shap_method = None, None, None

if SHAP_OK:
    try:
        if best_name == "Random Forest":
            explainer = shap.TreeExplainer(best_model.named_steps["rf"])
            Xt = best_model[:-1].transform(X_expl)
            shap_values = np.asarray(explainer.shap_values(Xt))
            shap_base = float(np.ravel(explainer.expected_value)[0]); shap_method = "SHAP TreeExplainer (exact)"
        else:
            background = shap.kmeans(X_train, 25)
            f = lambda a: best_model.predict(pd.DataFrame(a, columns=fe.FEATURES))
            explainer = shap.KernelExplainer(f, background)
            X_expl = X_expl.iloc[:80]
            shap_values = np.asarray(explainer.shap_values(X_expl, nsamples=200, silent=True))
            shap_base = float(np.ravel(explainer.expected_value)[0]); shap_method = "SHAP KernelExplainer (approximate, 25-cluster background)"
    except Exception as exc:
        print("SHAP failed, using the built-in explainer instead:", exc); shap_values = None

if shap_values is None:      # fallback: our own permutation-Shapley (same method the API uses)
    X_expl = X_expl.iloc[:60]
    base_series = X_train.mean()
    res = [explain_core.permutation_shapley(best_model, X_expl.iloc[i], base_series, fe.FEATURES, n_permutations=30, seed=SEED) for i in range(len(X_expl))]
    shap_values = np.array([[r["contributions"][f] for f in fe.FEATURES] for r in res])
    shap_base = float(np.mean([r["base_value"] for r in res])); shap_method = "built-in permutation-sampling Shapley (30 orderings)"
print("Explanation method:", shap_method, "| rows explained:", len(X_expl), "| base value: %.2f °C" % shap_base)

mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=fe.FEATURES).sort_values(ascending=False)
if SHAP_OK and "SHAP" in shap_method:
    plt.figure(); shap.summary_plot(shap_values, X_expl, feature_names=fe.FEATURES, show=False); plt.title("SHAP summary (beeswarm)")
    save_fig("xai_02_shap_summary")
fig, ax = plt.subplots(figsize=(8, 6)); mean_abs[::-1].plot.barh(ax=ax, color="#14b8a6")
ax.set(title="Mean |SHAP value| — average impact on predicted Tmax", xlabel="°C"); save_fig("xai_03_shap_bar")

In [ ]:
# 8.4 Local explanations for individual predictions (same lightweight method as the live API)
base_series = X_train.mean()
order = np.argsort(pred_test)
picks = {"Hottest predicted day": order[-1], "Typical day": order[len(order) // 2], "Coolest predicted day": order[0]}
local_examples = []
for label, i in picks.items():
    res = explain_core.permutation_shapley(best_model, X_test.iloc[i], base_series, fe.FEATURES, n_permutations=150, seed=SEED)
    contrib = pd.Series(res["contributions"]).sort_values(key=np.abs, ascending=False)
    local_examples.append({"label": label, "city": test_df["city"].iloc[i], "date": str(test_df["target_date"].iloc[i].date()),
                           "actual_c": float(y_test.iloc[i]), "predicted_c": float(pred_test[i]), "base_value_c": res["base_value"],
                           "contributions": {k: float(v) for k, v in contrib.head(8).items()}})
    print(f"\n{label}: {test_df['city'].iloc[i]} → {test_df['target_date'].iloc[i].date()} | actual {y_test.iloc[i]:.1f} °C | predicted {pred_test[i]:.1f} °C | base {res['base_value']:.1f} °C")
    print(contrib.head(6).round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, ex in zip(axes, local_examples):
    s = pd.Series(ex["contributions"])[::-1]
    ax.barh(s.index, s.values, color=["#ef4444" if v > 0 else "#3b82f6" for v in s.values])
    ax.set_title(f"{ex['label']}\n{ex['city']} {ex['date']} — pred {ex['predicted_c']:.1f} °C"); ax.set_xlabel("contribution (°C)")
save_fig("xai_04_local_explanations")

# Sanity check: does our lightweight method agree with SHAP for the same row?
if SHAP_OK and "SHAP" in shap_method:
    i0 = 0
    ours = explain_core.permutation_shapley(best_model, X_expl.iloc[i0], base_series, fe.FEATURES, n_permutations=150, seed=SEED)["contributions"]
    rho = pd.Series(ours).corr(pd.Series(dict(zip(fe.FEATURES, shap_values[i0]))), method="spearman")
    print(f"\nAgreement between the API's explainer and SHAP on one row (Spearman ρ): {rho:.2f}")

### How to read these plots (for the viva)
* **Bars to the right / red** = the feature pushed the predicted temperature *up* relative to an average day; **left / blue** = pushed it *down*.
* Expect *recent temperature* features to dominate (persistence), and humidity / cloud / radiation / season to adjust the forecast. Check the printed top-5 lists above to see what **your** run found — do not assume.
* Explanations describe the **model's behaviour**, not guaranteed physical causation.

## Section 9 — Temperature anomalies and heat categories

**Temperature anomaly = Predicted temperature − historical seasonal normal.**

* The *seasonal normal* is the mean daily-maximum temperature for that calendar day in **1991–2020** (the WMO climate-normal period).
* Data product: **ERA5-Land** reanalysis through the Open-Meteo *Historical Weather API* (`models=era5_land`; ~0.1° ≈ 9–11 km grid, hourly, 1950-present, produced by ECMWF/Copernicus). One consistent product is used for the whole baseline.
* A single calendar day has only 30 values, so we pool a **±7-day window (15 days × 30 years = 450 values)** around each day-of-year. From these we get the mean (normal), standard deviation, and the **90th and 95th percentile thresholds**.
* **Product consistency:** the model's target comes from the Open-Meteo forecast archive, not ERA5-Land. We measure the average difference between both products over the overlap (2022 → today) per city and shift the normals by it, so anomalies compare like with like. The raw ERA5-Land normal and the offset are kept in the CSV.

**Heat categories are project-defined, not official IMD warnings:**
`Normal` (< normal + 1.5 °C) · `Elevated` (≥ +1.5 °C anomaly) · `High` (≥ 90th percentile) · `Severe` (≥ 95th percentile).

In [ ]:
NORMAL_START, NORMAL_END, WINDOW_HALF, ERA_MODEL = date(1991, 1, 1), date(2020, 12, 31), 7, "era5_land"
APPLY_PRODUCT_OFFSET = True

def fetch_era_daily(city, start, end):
    c = fe.CITIES[city]; parts, meta = [], {}
    cur = start
    while cur <= end:
        stop = min(date(cur.year + 9, 12, 31), end)
        js = get_json(ERA5_URL, dict(latitude=c["lat"], longitude=c["lon"], start_date=cur.isoformat(), end_date=stop.isoformat(),
                                     daily="temperature_2m_max", models=ERA_MODEL, timezone=fe.TIMEZONE))
        unit = (js.get("daily_units") or {}).get("temperature_2m_max")
        if unit != "°C": raise ApiError(f"ERA unit problem: {unit}")
        parts.append(pd.DataFrame({"date": pd.to_datetime(js["daily"]["time"]), "era_tmax": js["daily"]["temperature_2m_max"]}))
        meta = {"grid_latitude": js.get("latitude"), "grid_longitude": js.get("longitude"), "elevation_m": js.get("elevation")}
        cur = stop + timedelta(days=1); time.sleep(1.0)
    return pd.concat(parts, ignore_index=True), meta

def climatology(era_df, half=WINDOW_HALF):
    s = era_df.dropna(subset=["era_tmax"]).copy(); s["doy"] = fe.day_of_year_365(s["date"])
    by_doy = {d: g["era_tmax"].to_numpy() for d, g in s.groupby("doy")}
    out = []
    for d in range(1, 366):
        idx = [((d - 1 + k) % 365) + 1 for k in range(-half, half + 1)]
        v = np.concatenate([by_doy[i] for i in idx if i in by_doy])
        out.append({"doy": d, "normal_tmax": v.mean(), "std_tmax": v.std(ddof=1), "p90_tmax": np.percentile(v, 90),
                    "p95_tmax": np.percentile(v, 95), "n_samples": len(v)})
    return pd.DataFrame(out)

normal_frames, era_meta, offsets = [], {}, {}
overlap_end = min(TRAIN_END, date.today() - timedelta(days=8))     # ERA5 lags ~5 days
for city in fe.CITIES:
    era_base, meta = fetch_era_daily(city, NORMAL_START, NORMAL_END)
    clim = climatology(era_base); clim.insert(0, "city", city)
    era_recent, _ = fetch_era_daily(city, TRAIN_START, overlap_end)
    both = daily[daily["city"] == city][["date", "tmax"]].merge(era_recent, on="date").dropna()
    diff = both["tmax"] - both["era_tmax"]
    offsets[city] = {"mean_offset_c": float(diff.mean()), "rmsd_c": float(np.sqrt((diff ** 2).mean())), "n_days": int(len(both)),
                     "monthly_offset_c": diff.groupby(both["date"].dt.month).mean().round(2).to_dict()}
    era_meta[city] = {**meta, "baseline_days": int(era_base["era_tmax"].notna().sum())}
    clim["normal_tmax_era5land_raw"] = clim["normal_tmax"]
    clim["product_offset_c"] = offsets[city]["mean_offset_c"] if APPLY_PRODUCT_OFFSET else 0.0
    for col in ("normal_tmax", "p90_tmax", "p95_tmax"): clim[col] = clim[col] + clim["product_offset_c"]
    normal_frames.append(clim)
    print(f"{city:<10} baseline days {era_meta[city]['baseline_days']:,} | product offset (forecast archive − ERA5-Land) "
          f"{offsets[city]['mean_offset_c']:+.2f} °C (RMSD {offsets[city]['rmsd_c']:.2f}, n={offsets[city]['n_days']})")

normals = pd.concat(normal_frames, ignore_index=True)
assert normals.groupby("city").size().eq(365).all() and normals[["normal_tmax", "p90_tmax", "p95_tmax"]].notna().all().all()
normals.round(3).to_csv(MODEL_DIR / "city_climate_normals.csv", index=False)
display(normals[normals["doy"].isin([15, 105, 135, 195, 300])].round(1))

In [ ]:
# Visualise the normals + percentile thresholds, and the seasonal product offset
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
for ax, (city, g) in zip(axes.ravel(), normals.groupby("city")):
    ax.fill_between(g["doy"], g["normal_tmax"], g["p95_tmax"], color="#fecaca", alpha=0.6, label="normal → 95th pct")
    ax.plot(g["doy"], g["normal_tmax"], color="#0f766e", label="1991–2020 normal"); ax.plot(g["doy"], g["p90_tmax"], "--", color="#f59e0b", label="90th pct")
    ax.plot(g["doy"], g["p95_tmax"], "--", color="#ef4444", label="95th pct"); ax.set_title(city)
axes[0, 0].legend(fontsize=7); fig.suptitle("City climate normals of daily Tmax (ERA5-Land, window ±7 days)", fontweight="bold")
save_fig("climate_01_normals")

fig, ax = plt.subplots(figsize=(9, 4))
for city in fe.CITIES:
    m = pd.Series(offsets[city]["monthly_offset_c"]); ax.plot(m.index, m.values, marker="o", label=city, color=palette[city])
ax.axhline(0, color="k", lw=0.8); ax.set(title="Product offset by month: forecast-archive Tmax − ERA5-Land Tmax (2022→)", xlabel="month", ylabel="°C"); ax.legend(ncol=3)
save_fig("climate_02_product_offset")

In [ ]:
# Anomalies and categories for the TEST period using the selected model
te = test_df[["city", "date", "target_date", fe.TARGET]].copy(); te["predicted"] = pred_test
te["doy"] = fe.day_of_year_365(te["target_date"])
te = te.merge(normals[["city", "doy", "normal_tmax", "p90_tmax", "p95_tmax"]], on=["city", "doy"], how="left")
te["anomaly_pred"] = te["predicted"] - te["normal_tmax"]; te["anomaly_actual"] = te[fe.TARGET] - te["normal_tmax"]
te["cat_pred"] = [fe.classify_heat(p, n, a, b)["label"] for p, n, a, b in zip(te["predicted"], te["normal_tmax"], te["p90_tmax"], te["p95_tmax"])]
te["cat_actual"] = [fe.classify_heat(p, n, a, b)["label"] for p, n, a, b in zip(te[fe.TARGET], te["normal_tmax"], te["p90_tmax"], te["p95_tmax"])]

print("Predicted heat categories on the test set:"); display(pd.crosstab(te["city"], te["cat_pred"]))
print("Confusion matrix (rows = category from ACTUAL Tmax, columns = category from PREDICTED Tmax):")
display(pd.crosstab(te["cat_actual"], te["cat_pred"]))

from sklearn.metrics import precision_score, recall_score, f1_score
ev_true, ev_pred = (te[fe.TARGET] >= te["p90_tmax"]), (te["predicted"] >= te["p90_tmax"])
heat_event = {"definition": "event = Tmax >= city/day 90th percentile of the 1991-2020 climatology",
              "n_actual_events": int(ev_true.sum()), "n_test_days": int(len(te)),
              "precision": float(precision_score(ev_true, ev_pred, zero_division=0)),
              "recall": float(recall_score(ev_true, ev_pred, zero_division=0)),
              "f1": float(f1_score(ev_true, ev_pred, zero_division=0))}
print("\nHeat-event detection on test set:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in heat_event.items()})
print("Mean anomaly (predicted): %+.2f °C | MAE of anomaly = MAE of temperature: %.2f °C" % (te["anomaly_pred"].mean(), (te["anomaly_pred"] - te["anomaly_actual"]).abs().mean()))

city_show = "Delhi"; g = te[te["city"] == city_show].sort_values("target_date")
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(g["target_date"], g[fe.TARGET], color="#64748b", lw=1, label="actual (model-derived)"); ax.plot(g["target_date"], g["predicted"], color="#0d9488", lw=1.4, label="predicted")
ax.plot(g["target_date"], g["normal_tmax"], color="#0f766e", ls=":", label="1991–2020 normal"); ax.plot(g["target_date"], g["p90_tmax"], "--", color="#f59e0b", label="90th pct"); ax.plot(g["target_date"], g["p95_tmax"], "--", color="#ef4444", label="95th pct")
ax.set(title=f"{city_show}: predicted vs actual Tmax against climatology (test period)", ylabel="°C"); ax.legend(ncol=5, fontsize=8)
save_fig("climate_03_anomaly_example")

## Section 10 — Model export
Everything the website needs is written to `climatesense_output/backend/models/`:

| File | Purpose |
|---|---|
| `best_model.joblib` | the fitted scikit-learn Pipeline selected by validation MAE |
| `model_metrics.json` | actual MAE / MSE / R² (train, validation, test), baseline, interval, per-city scores |
| `feature_importance.json` | RF importance, permutation importance, SHAP summary data, local examples |
| `feature_schema.json` | feature names/order/units, training statistics and baseline, feature-module version |
| `training_metadata.json` | data source, periods, hyper-parameters, environment, limitations |
| `city_climate_normals.csv` | 1991–2020 city normals and p90/p95 thresholds by day-of-year |

In [ ]:
# Feature importance + explanation JSON
shap_rows = [{"x": {f: round(float(X_expl.iloc[i][f]), 3) for f in fe.FEATURES},
              "shap": {f: round(float(shap_values[i][j]), 4) for j, f in enumerate(fe.FEATURES)}} for i in range(len(X_expl))]
importance_doc = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "best_model": best_name,
    "units": "°C",
    "rf_impurity": {"description": "Random Forest impurity-based importance (sums to 1)", "values": rf_imp.round(5).to_dict()},
    "permutation": {"description": f"Increase in MAE (°C) on the TEST set when a feature is shuffled; model = {best_name}; 10 repeats",
                    "mean": perm["mean"].round(5).to_dict(), "std": perm["std"].round(5).to_dict()},
    "shap": {"method": shap_method, "base_value_c": shap_base, "n_rows": len(X_expl), "mean_abs": mean_abs.round(5).to_dict(), "rows": shap_rows},
    "local_examples": local_examples,
    "feature_labels": {f: {"label": fe.FEATURE_INFO[f][0], "unit": fe.FEATURE_INFO[f][1]} for f in fe.FEATURES},
}
(MODEL_DIR / "feature_importance.json").write_text(json.dumps(jsonable(importance_doc), indent=1, ensure_ascii=False))

# Feature schema (the contract between training and the live API)
stats = {f: {"min": float(X_train[f].min()), "max": float(X_train[f].max()), "mean": float(X_train[f].mean()),
             "median": float(X_train[f].median()), "std": float(X_train[f].std())} for f in fe.FEATURES}
schema_doc = {
    "features": fe.FEATURES, "target": fe.TARGET, "details": fe.feature_schema_entries(),
    "feature_module_version": fe.FEATURE_MODULE_VERSION, "feature_module_sha256": FE_SHA256,
    "hourly_variables": fe.MODEL_HOURLY_VARIABLES, "expected_units": fe.EXPECTED_UNITS,
    "cutoff_definition": "Features use data of days <= D (local calendar days); target = max 2 m temperature of day D+1.",
    "training_cities": list(fe.CITIES), "feature_stats": stats, "baseline": {f: stats[f]["mean"] for f in fe.FEATURES},
}
(MODEL_DIR / "feature_schema.json").write_text(json.dumps(jsonable(schema_doc), indent=2, ensure_ascii=False))

# Heat-event results added to the metrics file
metrics_doc["heat_event_detection"] = heat_event
metrics_doc["heat_categories"] = {"levels": fe.HEAT_LEVELS, "disclaimer": fe.HEAT_DISCLAIMER}
metrics_doc["figures"] = sorted(p.name for p in FIG_DIR.glob("*.png"))
(MODEL_DIR / "model_metrics.json").write_text(json.dumps(jsonable(metrics_doc), indent=2, ensure_ascii=False))

# Training metadata
training_doc = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset": {**{k: dataset_meta[k] for k in ["source", "source_url", "retrieval_date_utc", "observation_period", "cities", "units",
                                                "hourly_records", "daily_records", "model_rows", "nature_of_values"]},
                "period": dataset_meta["observation_period"]},
    "training": {"seed": SEED, "cv": f"{len(cv_splits)}-fold expanding-window by unique date, 1-day embargo", "scoring": "neg_mean_absolute_error",
                 "search": search_info, "split": split_info, "selected_model": best_name},
    "climate_normals": {"baseline_period": "1991-2020", "product": "ERA5-Land via Open-Meteo Historical Weather API (models=era5_land)",
                        "variable": "temperature_2m_max (daily)", "window_days": 2 * WINDOW_HALF + 1, "percentiles": [90, 95],
                        "product_offset_applied": APPLY_PRODUCT_OFFSET, "product_offsets": offsets, "grid": era_meta,
                        "resolution": "approx. 0.1° (≈9-11 km)", "documentation": "https://open-meteo.com/en/docs/historical-weather-api"},
    "environment": {"python": platform.python_version(), "scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
                    "shap": shap.__version__ if SHAP_OK else None, "platform": platform.platform()},
    "limitations": [
        "Training data are archived weather-model values (not station observations); the target is model-derived.",
        "About four years of data: enough to learn the seasonal cycle, not to represent rare extremes or climate trends.",
        "Latitude/longitude identify only six cities; the model is not validated for other locations.",
        "Heat categories are project-defined and are not IMD warnings.",
        "The daily regression model forecasts weather; it does not prove greenhouse-gas causation.",
    ],
}
(MODEL_DIR / "training_metadata.json").write_text(json.dumps(jsonable(training_doc), indent=2, ensure_ascii=False))

joblib.dump(best_model, MODEL_DIR / "best_model.joblib", compress=3)
(MODEL_DIR / "RUNTIME_VERSIONS.txt").write_text(f"scikit-learn=={sklearn.__version__}\npandas=={pd.__version__}\nnumpy=={np.__version__}\n")
for p in sorted(MODEL_DIR.iterdir()):
    if p.is_file(): print(f"{p.name:<28}{p.stat().st_size / 1024:>10.1f} KB")

In [ ]:
# Verify the export: reload from disk, predict, and check that outputs match the in-memory model.
reloaded = joblib.load(MODEL_DIR / "best_model.joblib")
assert np.allclose(reloaded.predict(X_test.head(50)), best_model.predict(X_test.head(50)))
assert json.loads((MODEL_DIR / "feature_schema.json").read_text())["features"] == fe.FEATURES
print("✔ Exported model reproduces the in-memory predictions; feature schema matches feature_engineering.FEATURES")
print(f"\nFINAL SUMMARY — selected model: {best_name}")
print(f"  validation MAE {results[best_name]['validation']['mae']:.3f} °C | MSE {results[best_name]['validation']['mse']:.3f} | R² {results[best_name]['validation']['r2']:.3f}")
print(f"  test       MAE {results[best_name]['test']['mae']:.3f} °C | MSE {results[best_name]['test']['mse']:.3f} | R² {results[best_name]['test']['r2']:.3f}")
print(f"  persistence baseline test MAE {baseline['test']['mae']:.3f} °C")
print(f"\nPin this in backend/requirements.txt:  scikit-learn=={sklearn.__version__}")

### Download the artifacts
The next cell zips `backend/models/` (model, JSON files, normals, figures) and `data/` (raw CSV, metadata). In Colab it starts the download automatically. Unzip the archive in the **project root** so the files land in `backend/models/` and `data/`.

In [ ]:
zip_path = Path("climatesense_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(ROOT))
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB) with {len(zf.namelist())} files")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab - find the zip next to the notebook.")

## Reproducibility notes & limitations
* **Seeds:** `SEED = 42` is used for NumPy, Python, the Random Forest, permutation importance and sampling.
* **Data drift:** the API archive is extended every day, so a re-run later will include more recent dates and give slightly different numbers. The retrieval date and observation period are stored in `dataset_metadata.json`.
* **What the numbers mean:** MAE is the typical error in °C. Always compare with the persistence baseline above.
* **Limitations:** model-derived data, short history (≈4 years), six cities, project-defined heat categories, and a weather-scale model that cannot attribute temperature change to greenhouse gases.